In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types  import *
from pyspark.sql.functions import col


In [4]:
spark=SparkSession.builder \
      .appName("E-commerce") \
       .master("yarn") \
       .getOrCreate()
print("Session Created")
       

26/03/05 15:41:27 WARN Utils: Your hostname, nafisashaik resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/05 15:41:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/05 15:41:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/05 15:41:41 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
Session Created


In [8]:
raw_path1="hdfs:///ecommerce/raw/olist_orders_dataset.csv"
df_raw1=spark.read \
    .option("header","true") \
    .option("infer schema","true") \
    .csv(raw_path1)
print("orders table loaded successfully")
df_raw1.show(5)

    

orders table loaded successfully


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [10]:
raw_path2="hdfs:///ecommerce/raw/olist_order_items_dataset.csv"
df_raw2=spark.read \
    .option("header","true") \
    .option("infer schema","true") \
    .csv(raw_path2)
print("order items list data loaded successfully")
df_raw2.show(5)

    

order items list data loaded successfully


+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.90|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.90|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.00|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.90|        18.14|
+--------------------+-------------+----

In [11]:
df_raw1.describe()

DataFrame[summary: string, order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: string, order_approved_at: string, order_delivered_carrier_date: string, order_delivered_customer_date: string, order_estimated_delivery_date: string]

In [21]:
#total records in orders
df_raw1.count()

99441

In [22]:
#total records in lists
df_raw2.count()

112650

In [12]:
df_raw2.describe()

DataFrame[summary: string, order_id: string, order_item_id: string, product_id: string, seller_id: string, shipping_limit_date: string, price: string, freight_value: string]

In [13]:
# Join orders and order_items using order_id

df_joined = df_raw1.join(
    df_raw2,
    df_raw1.order_id == df_raw2.order_id,
    "inner"
)
print("Tables joined successfully")
df_joined.show(5)

Tables joined successfully


[Stage 13:>                                                         (0 + 2) / 2]

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00018f77f2f0320c5...|f6dd3ec061db4e398...|   delivered|     2017-04-26 1

In [25]:
# handle null values
df_joined.fillna(0)

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: string, order_delivered_carrier_date: string, order_delivered_customer_date: string, order_estimated_delivery_date: string, order_id: string, order_item_id: string, product_id: string, seller_id: string, shipping_limit_date: string, price: double, freight_value: double, revenue: double]

In [26]:
from pyspark.sql.functions import to_timestamp

df_joined = df_joined.withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp"))
)

In [27]:
from pyspark.sql.functions import sum, month, desc, avg
#calc total revenue
df_joined = df_joined.withColumn(
    "revenue",
    col("price") + col("freight_value")
)

total_revenue = df_joined.select(
    sum("revenue").alias("total_revenue")
)

total_revenue.show()

[Stage 43:=============================>                            (1 + 1) / 2]

+--------------------+
|       total_revenue|
+--------------------+
|1.5843553239999255E7|
+--------------------+



In [29]:
#revenue per month
revenue_per_month = df_joined.groupBy(
    month("order_purchase_timestamp").alias("month")
).agg(sum("revenue").alias("monthly_revenue"))

revenue_per_month.show()

[Stage 51:>                                                         (0 + 2) / 2]

+-----+------------------+
|month|   monthly_revenue|
+-----+------------------+
|   12| 863566.8500000024|
|    1|1244490.3800000027|
|    6|        1525640.15|
|    3|1587175.4100000006|
|    5|1735972.7700000033|
|    9| 720920.1200000022|
|    4|1572120.2800000012|
|    8|1671513.0699999973|
|    7| 1643699.649999996|
|   10| 826121.2100000036|
|   11|1179143.7700000012|
|    2|1273189.5800000026|
+-----+------------------+



In [30]:
# top 5 selling products
top_products = df_joined.groupBy(
    "product_id"
).agg(
    sum("order_item_id").alias("total_sold")
).orderBy(
    desc("total_sold")
).limit(5)

top_products.show()

[Stage 55:=============================>                            (1 + 1) / 2]

+--------------------+----------+
|          product_id|total_sold|
+--------------------+----------+
|422879e10f4668299...|     793.0|
|aca2eb7d00ea1a7b8...|     640.0|
|368c6c730842d7801...|     551.0|
|53759a2ecddad2bb8...|     545.0|
|99a4788cb24856965...|     542.0|
+--------------------+----------+



In [31]:
from pyspark.sql.functions import col, sum, avg

# convert columns to numbers
df_joined = df_joined.withColumn("price", col("price").cast("double"))
df_joined = df_joined.withColumn("freight_value", col("freight_value").cast("double"))

# calculate total value per order
order_total = df_joined.groupBy(df_raw1.order_id).agg(
    sum(col("price") + col("freight_value")).alias("order_total")
)

# calculate average order value
avg_order_value = order_total.agg(
    avg("order_total").alias("avg_order_value")
)

avg_order_value.show()

[Stage 59:=============================>                            (1 + 1) / 2]

+------------------+
|   avg_order_value|
+------------------+
|160.57763809214612|
+------------------+



In [32]:
# Save results to HDFS in Parquet format

revenue_per_month.write \
    .mode("overwrite") \
    .parquet("hdfs:///ecommerce/analytics/revenue_per_month")

top_products.write \
    .mode("overwrite") \
    .parquet("hdfs:///ecommerce/analytics/top_products")

avg_order_value.write \
    .mode("overwrite") \
    .parquet("hdfs:///ecommerce/analytics/avg_order_value")

print("Data saved  in /ecommerce/analytics/")

[Stage 74:=============================>                            (1 + 1) / 2]

Data saved successfully in /ecommerce/analytics/
